# Лабораторная работа №2: Детекция и сегментация объектов

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Проверить разметку и распределения объектов.
- Дообучить компактный segmentation baseline.
- Сравнить аугментации и проанализировать ошибки.


## Теоретическая часть

В детекции модель предсказывает классы и bounding boxes, в сегментации — метку каждого пикселя. Для класса $c$ качество сегментации измеряет $IoU_c=|P_c\cap G_c|/|P_c\cup G_c|$. Для детекции mAP усредняет AP по классам и порогам IoU.

![Segmentation example](https://pytorch.org/vision/main/_images/sphx_glr_plot_visualization_utils_008.png)


### Математическая постановка

Составная функция потерь может иметь вид
$$\mathcal L=\mathcal L_{cls}+\lambda_{box}\mathcal L_{box}+\lambda_{mask}\mathcal L_{mask}.$$
Разбиение должно выполняться по исходным объектам/сериям, иначе соседние кадры дают утечку.


### Материалы

- [Torchvision segmentation models](https://pytorch.org/vision/stable/models.html#semantic-segmentation)
- [COCO metrics](https://cocodataset.org/#detection-eval)
- [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Проверить разметку и распределения объектов.
2. Дообучить компактный segmentation baseline.
3. Сравнить аугментации и проанализировать ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** LRASPP-MobileNetV3 или FCN-ResNet50, mixed precision.
- **Ограничение данных:** Oxford-IIIT Pet или до 2 000 изображений; в smoke-режиме — синтетический датасет.
- **Fallback:** замороженный backbone и 256 px.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [15]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только smoke-проверка; не отчётный результат

# === Папка артефактов рядом с notebook ===
ARTIFACTS = Path.cwd() / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"

_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "artifacts": str(ARTIFACTS), "data_root": str(DATA_ROOT)})

{'device': 'cuda', 'fast_dev_run': False, 'artifacts': 'c:\\Users\\Lenovo\\Projects\\PythonSOTA\\lab2\\artifacts', 'data_root': 'c:\\Users\\Lenovo\\Projects\\data'}


In [16]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Маски Oxford-IIIT Pet

*Вес: 1 балл.*

**Постановка.** Сегментатор обучается на реальной пиксельной разметке.

Выполните следующие действия:
1. Загрузите trainval Oxford-IIIT Pet и совместно масштабируйте изображение и маску.
2. Убедитесь, что интерполяция маски сохраняет классы.

**Результат.** Датасет ds и преобразования img_tf, mask_tf.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [17]:
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import v2
import torchvision.transforms.functional as F
from torchvision.tv_tensors import Mask

IMG_SIZE = 256

# Совместные преобразования: изображение — bilinear, маска — nearest
train_tf = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Для маски отдельно, чтобы гарантировать целочисленные классы
def mask_tf(mask):
    mask = F.resize(mask, [IMG_SIZE, IMG_SIZE], interpolation=F.InterpolationMode.NEAREST)
    return Mask(torch.as_tensor(np.array(mask), dtype=torch.long))

ds = OxfordIIITPet(
    root=ARTIFACTS / "oxford-iiit-pet",
    split="trainval",
    target_types="segmentation",
    download=True,
    transform=train_tf,
    target_transform=mask_tf,
)

img, mask = ds[0]
print("image:", img.shape, img.dtype)
print("mask:", mask.shape, mask.dtype)
print("уникальные классы маски:", torch.unique(mask))

image: torch.Size([3, 256, 256]) torch.float32
mask: torch.Size([256, 256]) torch.int64
уникальные классы маски: tensor([1, 2, 3])


### Разделение и кодирование меток

*Вес: 1 балл.*

**Постановка.** Оценка на validation требует непересекающихся индексов и корректных категориальных меток.

Выполните следующие действия:
1. Разделите trainval на train и validation.
2. Перекодируйте исходные метки 1–3 в 0–2 и соберите загрузчики.

**Результат.** train_ds, val_ds, loader, val_loader и normalize_mask.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [18]:
from torch.utils.data import DataLoader, Subset

# Перекодировка 1–3 → 0–2
def normalize_mask(mask):
    return (mask - 1).long()

class PetSegDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds):
        self.base = base_ds
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        img, mask = self.base[idx]
        return img, normalize_mask(mask).squeeze(0)

full = PetSegDataset(ds)

# Воспроизводимое разделение trainval → train / val
n = len(full)
n_val = int(0.2 * n)
n_train = n - n_val
g = torch.Generator().manual_seed(SEED)
indices = torch.randperm(n, generator=g).tolist()
train_idx, val_idx = indices[:n_train], indices[n_train:]

# Smoke-режим: маленькие подвыборки, чтобы не ждать
if FAST_DEV_RUN:
    train_idx = train_idx[:300]
    val_idx   = val_idx[:100]

train_ds = Subset(full, train_idx)
val_ds   = Subset(full, val_idx)

# num_workers=0 — обязательно для Jupyter/Windows,
# иначе DataLoader при итерации спавнит процессы, которые
# повторно исполняют ячейку и создают вложенные DataLoader'ы (форк-бомба).
# Для полноценного обучения вынесите цикл в .py-скрипт с if __name__ == "__main__"
# и поставьте num_workers=4 + persistent_workers=True.
NUM_WORKERS = 0

loader_kwargs = dict(
    batch_size=16,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)

loader     = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader = DataLoader(val_ds,   shuffle=False, **loader_kwargs)

print(f"train: {len(train_ds)}, val: {len(val_ds)}, num_workers: {NUM_WORKERS}")

train: 2944, val: 736, num_workers: 0


### Сегментатор и IoU

*Вес: 1 балл.*

**Постановка.** Пиксельная модель предсказывает логиты для каждого класса.

Выполните следующие действия:
1. Замените головы LRASPP на трёхклассовые.
2. Реализуйте IoU по пересечению и объединению масок.

**Результат.** Объект model и функция mean_iou.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [19]:
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

model = lraspp_mobilenet_v3_large(weights="DEFAULT", weights_backbone="DEFAULT")
# Замена головы на 3 класса
model.classifier.low_classifier = torch.nn.Conv2d(40, 3, kernel_size=1)
model.classifier.high_classifier = torch.nn.Conv2d(128, 3, kernel_size=1)
model.to(DEVICE)

def confusion_matrix(pred, target, num_classes=3):
    """pred, target: (N, H, W) int64"""
    mask = (target >= 0) & (target < num_classes)
    idx = num_classes * target[mask].to(torch.int64) + pred[mask].to(torch.int64)
    cm = torch.bincount(idx, minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return cm

def mean_iou(cm):
    intersection = torch.diag(cm).float()
    union = cm.sum(dim=1) + cm.sum(dim=0) - torch.diag(cm)
    iou = intersection / union.clamp_min(1e-6)
    return iou.mean().item(), iou

print(model.classifier)

LRASPPHead(
  (cbr): Sequential(
    (0): Conv2d(960, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (scale): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Conv2d(960, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): Sigmoid()
  )
  (low_classifier): Conv2d(40, 3, kernel_size=(1, 1), stride=(1, 1))
  (high_classifier): Conv2d(128, 3, kernel_size=(1, 1), stride=(1, 1))
)


### Оптимизация сегментатора

*Вес: 1 балл.*

**Постановка.** Cross-entropy на масках задаёт обучающий сигнал каждому пикселю.

Выполните следующие действия:
1. Реализуйте шаг AdamW с обратным распространением.
2. Обучите модель установленное число эпох.

**Результат.** Функция train_one_epoch и массив history реальных значений loss.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [20]:
EPOCHS = 8
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)

history = []
ckpt_last = ARTIFACTS / "segmentation_last.pt"

# --- resume, если чекпоинт есть ---
start_epoch = 0
if not FAST_DEV_RUN and ckpt_last.exists():
    ckpt = torch.load(ckpt_last, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    history = ckpt["history"]
    start_epoch = ckpt["epoch"] + 1
    print(f"resume from epoch {start_epoch+1}")

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)["out"]
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

for epoch in range(start_epoch, EPOCHS):
    loss = train_one_epoch(model, loader, optimizer, criterion, DEVICE)
    history.append(loss)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  loss={loss:.4f}")

    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "history": history,
    }, ckpt_last)

epoch 01/8  loss=0.3926
epoch 02/8  loss=0.2351
epoch 03/8  loss=0.2085
epoch 04/8  loss=0.1942
epoch 05/8  loss=0.1817
epoch 06/8  loss=0.1721
epoch 07/8  loss=0.1649
epoch 08/8  loss=0.1596


### Валидационный IoU

*Вес: 1 балл.*

**Постановка.** Матрица ошибок по пикселям позволяет вычислить классовый IoU.

Выполните следующие действия:
1. Накопите confusion matrix по validation.
2. Сохраните mIoU и checkpoint исходной модели.

**Результат.** segmentation.pt и val_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [21]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    cm = torch.zeros(3, 3, dtype=torch.int64)
    for xb, yb in loader:
        xb = xb.to(device)
        pred = model(xb)["out"].argmax(dim=1).cpu()
        cm += confusion_matrix(pred, yb, num_classes=3)
    miou, per_class = mean_iou(cm)
    return miou, per_class, cm

val_miou, val_per_class, val_cm = evaluate(model, val_loader, DEVICE)
print(f"val mIoU: {val_miou:.4f}")
print("per-class IoU:", val_per_class.numpy().round(4))

torch.save(model.state_dict(), ARTIFACTS / "segmentation.pt")
val_metrics = {
    "val_miou": round(val_miou, 4),
    "per_class_iou": {str(i): round(float(v), 4) for i, v in enumerate(val_per_class)},
    "confusion_matrix": val_cm.tolist(),
}
with open(ARTIFACTS / "val_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=2)

val mIoU: 0.7732
per-class IoU: [0.8712 0.9332 0.5154]


### Совместное отражение и повторное обучение

*Вес: 1 балл.*

**Постановка.** Геометрическое преобразование изображения без идентичного преобразования маски портит разметку.

Выполните следующие действия:
1. Реализуйте синхронное горизонтальное отражение.
2. Обучите новый сегментатор при том же бюджете и сравните validation mIoU.

**Результат.** segmentation_aug.pt и augmentation_comparison.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [22]:
from torch.utils.data import DataLoader, Subset
import time

IMG_SIZE = 256

def train_transform(img, mask, augment=True):
    mask_t = torch.from_numpy(np.array(mask).astype(np.int64))

    img = v2.functional.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    mask_t = F.resize(
        mask_t.unsqueeze(0),
        [IMG_SIZE, IMG_SIZE],
        interpolation=F.InterpolationMode.NEAREST,
    ).squeeze(0)

    if augment and torch.rand(1).item() < 0.5:
        img    = F.hflip(img)
        mask_t = torch.flip(mask_t, dims=[-1])

    img = v2.functional.to_image(img)
    img = v2.functional.to_dtype(img, torch.float32, scale=True)
    img = v2.functional.normalize(
        img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    )
    return img, mask_t

class PetSegAug(torch.utils.data.Dataset):
    def __init__(self, base_ds, train: bool):
        self.base = base_ds
        self.train = train
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        img, mask = self.base[idx]
        img, mask = train_transform(img, mask, augment=self.train)
        return img, (mask - 1).long()

ds_aug = OxfordIIITPet(
    root=ARTIFACTS / "oxford-iiit-pet",
    split="trainval",
    target_types="segmentation",
    download=False,
    transform=None,
    target_transform=None,
)

_img, _mask = PetSegAug(ds_aug, train=True)[0]
assert _img.shape  == (3, IMG_SIZE, IMG_SIZE), _img.shape
assert _mask.shape == (IMG_SIZE, IMG_SIZE),    _mask.shape
assert _mask.dtype == torch.int64,             _mask.dtype
assert set(_mask.unique().tolist()).issubset({0, 1, 2}), _mask.unique().tolist()
print("sanity ok:", _img.shape, _mask.shape, _mask.unique().tolist())

train_aug = Subset(PetSegAug(ds_aug, train=True),  train_idx)
val_aug   = Subset(PetSegAug(ds_aug, train=False), val_idx)

loader_aug     = DataLoader(train_aug, batch_size=16, shuffle=True,
                            num_workers=0, pin_memory=torch.cuda.is_available())
val_loader_aug = DataLoader(val_aug,   batch_size=16, shuffle=False,
                            num_workers=0, pin_memory=torch.cuda.is_available())
print(f"train_aug: {len(train_aug)}, val_aug: {len(val_aug)}")

# --- seed control: тот же init, что у baseline ---
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_aug = lraspp_mobilenet_v3_large(weights="DEFAULT", weights_backbone="DEFAULT")
model_aug.classifier.low_classifier  = torch.nn.Conv2d(40,  3, kernel_size=1)
model_aug.classifier.high_classifier = torch.nn.Conv2d(128, 3, kernel_size=1)
model_aug.to(DEVICE)

optimizer_aug = torch.optim.AdamW(model_aug.parameters(), lr=1e-4, weight_decay=1e-4)

ckpt_aug_last = ARTIFACTS / "segmentation_aug_last.pt"
start_epoch_aug = 0
if not FAST_DEV_RUN and ckpt_aug_last.exists():
    ckpt = torch.load(ckpt_aug_last, map_location=DEVICE)
    model_aug.load_state_dict(ckpt["model"])
    optimizer_aug.load_state_dict(ckpt["optimizer"])
    start_epoch_aug = ckpt["epoch"] + 1
    print(f"resume aug from epoch {start_epoch_aug+1}")

LOG_EVERY = 20
n_batches = len(loader_aug)

for epoch in range(start_epoch_aug, EPOCHS):
    model_aug.train()
    total, n = 0.0, 0
    t_epoch = time.perf_counter()
    for i, (xb, yb) in enumerate(loader_aug, start=1):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer_aug.zero_grad(set_to_none=True)
        logits = model_aug(xb)["out"]
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer_aug.step()
        total += loss.item(); n += 1
        if i % LOG_EVERY == 0 or i == n_batches:
            print(f"[aug] epoch {epoch+1:02d}/{EPOCHS}  "
                  f"batch {i:03d}/{n_batches}  "
                  f"loss={total/n:.4f}  time={time.perf_counter()-t_epoch:.1f}s")
    print(f"[aug] epoch {epoch+1:02d}/{EPOCHS}  loss={total/n:.4f}")

    torch.save({
        "epoch": epoch,
        "model": model_aug.state_dict(),
        "optimizer": optimizer_aug.state_dict(),
    }, ckpt_aug_last)

val_miou_aug, per_class_aug, cm_aug = evaluate(model_aug, val_loader_aug, DEVICE)
print(f"[aug] val mIoU: {val_miou_aug:.4f}")

torch.save(model_aug.state_dict(), ARTIFACTS / "segmentation_aug.pt")
with open(ARTIFACTS / "augmentation_comparison.json", "w") as f:
    json.dump({
        "baseline_val_miou":  round(val_miou, 4),
        "augmented_val_miou": round(val_miou_aug, 4),
        "delta":              round(val_miou_aug - val_miou, 4),
    }, f, indent=2)
print("saved:", ARTIFACTS / "augmentation_comparison.json")

sanity ok: torch.Size([3, 256, 256]) torch.Size([256, 256]) [0, 1, 2]
train_aug: 2944, val_aug: 736
[aug] epoch 01/8  batch 020/184  loss=0.8748  time=5.2s
[aug] epoch 01/8  batch 040/184  loss=0.7037  time=10.3s
[aug] epoch 01/8  batch 060/184  loss=0.5989  time=15.0s
[aug] epoch 01/8  batch 080/184  loss=0.5331  time=19.5s
[aug] epoch 01/8  batch 100/184  loss=0.4878  time=24.0s
[aug] epoch 01/8  batch 120/184  loss=0.4563  time=28.6s
[aug] epoch 01/8  batch 140/184  loss=0.4315  time=33.1s
[aug] epoch 01/8  batch 160/184  loss=0.4111  time=38.1s
[aug] epoch 01/8  batch 180/184  loss=0.3948  time=43.0s
[aug] epoch 01/8  batch 184/184  loss=0.3926  time=44.1s
[aug] epoch 01/8  loss=0.3926
[aug] epoch 02/8  batch 020/184  loss=0.2559  time=5.0s
[aug] epoch 02/8  batch 040/184  loss=0.2502  time=9.9s
[aug] epoch 02/8  batch 060/184  loss=0.2458  time=14.9s
[aug] epoch 02/8  batch 080/184  loss=0.2465  time=19.6s
[aug] epoch 02/8  batch 100/184  loss=0.2446  time=24.2s
[aug] epoch 02/8  

### Сегментация малых объектов

*Вес: 1 балл.*

**Постановка.** Размер категории, заданный по train, позволяет отдельно изучать слабые места модели.

Выполните следующие действия:
1. Определите порог площади объекта только на train.
2. Вычислите foreground IoU на малых объектах validation для обеих моделей.

**Результат.** small_objects.json с размером подвыборки и IoU двух моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [23]:
# --- Площади foreground ТОЛЬКО по маске (без image decode/transform) ---
def mask_areas(base_ds, indices):
    """base_ds — сырой OxfordIIITPet (transform=None). indices — позиции в нём."""
    areas = []
    for i in indices:
        _, mask = base_ds[i]                                    # PIL-маска
        mask = F.resize(mask, [IMG_SIZE, IMG_SIZE],
                        interpolation=F.InterpolationMode.NEAREST)
        m = torch.from_numpy(np.array(mask).astype(np.int64))
        areas.append(int((m == 1).sum()))                       # class 1 = pet в исходной разметке
    return np.array(areas)

train_areas = mask_areas(ds_aug, train_idx)                    # порог — только по train
threshold = float(np.percentile(train_areas, 25))
print(f"порог площади (25-й перцентиль train): {threshold:.0f} px")

val_areas = mask_areas(ds_aug, val_idx)
small_idx = np.where(val_areas < threshold)[0].tolist()        # позиции в val_ds
assert len(small_idx) > 0, "пустая подвыборка — снизьте порог или проверьте seed"

small_ds     = Subset(val_ds, small_idx)
small_loader = DataLoader(small_ds, batch_size=8, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
print(f"малых объектов в val: {len(small_idx)} из {len(val_ds)}")

def foreground_iou(model, loader, device):
    model.eval()
    inter, union = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            pred = model(xb)["out"].argmax(dim=1).cpu()
            inter += ((pred == 0) & (yb == 0)).sum().item()
            union += ((pred == 0) | (yb == 0)).sum().item()
    return inter / max(union, 1)

fg_iou_base = foreground_iou(model,     small_loader, DEVICE)
fg_iou_aug  = foreground_iou(model_aug, small_loader, DEVICE)

print(f"foreground IoU (baseline): {fg_iou_base:.4f}")
print(f"foreground IoU (aug):      {fg_iou_aug:.4f}")

with open(ARTIFACTS / "small_objects.json", "w") as f:
    json.dump({
        "threshold_px":    int(threshold),
        "n_small_val":     len(small_idx),
        "baseline_fg_iou": round(fg_iou_base, 4),
        "aug_fg_iou":      round(fg_iou_aug, 4),
    }, f, indent=2)
print("saved:", ARTIFACTS / "small_objects.json")

порог площади (25-й перцентиль train): 11458 px
малых объектов в val: 164 из 736
foreground IoU (baseline): 0.7656
foreground IoU (aug):      0.7678
saved: c:\Users\Lenovo\Projects\PythonSOTA\lab2\artifacts\small_objects.json


### Отложенная оценка двух моделей

*Вес: 1 балл.*

**Постановка.** Test не должен влиять на обучение и выбор конфигурации.

Выполните следующие действия:
1. Загрузите test-раздел Oxford-IIIT Pet.
2. Оцените оба сохранённых сегментатора на одинаковых test-изображениях.

**Результат.** test_metrics.json с IoU на реальном test.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [24]:
ds_test = OxfordIIITPet(
    root=ARTIFACTS / "oxford-iiit-pet",
    split="test",
    target_types="segmentation",
    download=True,
    transform=train_tf,
    target_transform=mask_tf,
)
test_ds = PetSegDataset(ds_test)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False,
                         num_workers=0, pin_memory=torch.cuda.is_available())

test_miou_base, test_pc_base, _ = evaluate(model,     test_loader, DEVICE)
test_miou_aug,  test_pc_aug,  _ = evaluate(model_aug, test_loader, DEVICE)

test_metrics = {
    "test_miou_baseline":          round(test_miou_base, 4),
    "test_miou_aug":               round(test_miou_aug,  4),
    "test_per_class_iou_baseline": [round(float(x), 4) for x in test_pc_base],
    "test_per_class_iou_aug":      [round(float(x), 4) for x in test_pc_aug],
}
print(json.dumps(test_metrics, indent=2))
with open(ARTIFACTS / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

{
  "test_miou_baseline": 0.7768,
  "test_miou_aug": 0.7808,
  "test_per_class_iou_baseline": [
    0.8691,
    0.9301,
    0.5313
  ],
  "test_per_class_iou_aug": [
    0.87,
    0.9315,
    0.541
  ]
}


### Локализация ошибок маски

*Вес: 1 балл.*

**Постановка.** False positive и false negative выделяют различные режимы сбоя сегментатора.

Выполните следующие действия:
1. Посчитайте IoU, FP и FN для каждого изображения validation.
2. Сохраните наиболее сложные изображения и их индексы.

**Результат.** error_analysis.json с фактическими ошибками модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [25]:
@torch.no_grad()
def per_image_error(model, dataset, device):
    model.eval()
    rows = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        x = x.unsqueeze(0).to(device)
        pred = model(x)["out"].argmax(dim=1).squeeze(0).cpu()
        y = y.cpu()

        gt_area = int((y == 0).sum())                            # площадь GT-питомца
        pred_area = int((pred == 0).sum())

        inter = ((pred == 0) & (y == 0)).sum().item()
        union = ((pred == 0) | (y == 0)).sum().item()
        iou = inter / max(union, 1)
        fp = ((pred == 0) & (y != 0)).sum().item()
        fn = ((pred != 0) & (y == 0)).sum().item()
        rows.append({
            "idx": i, "iou": iou, "fp": fp, "fn": fn,
            "gt_area": gt_area, "pred_area": pred_area,
        })
    return rows

errors = per_image_error(model, val_ds, DEVICE)

# Разделяем аномалии разметки и реальные сложные случаи
anomalies  = [r for r in errors if r["gt_area"] < 100]
valid_rows = [r for r in errors if r["gt_area"] >= 100]
valid_sorted = sorted(valid_rows, key=lambda r: r["iou"])
hardest = valid_sorted[:10]

print(f"всего val: {len(errors)}")
print(f"аномалий разметки (GT pet < 100 px): {len(anomalies)}")
print(f"валидных для анализа: {len(valid_rows)}")
print("Топ-3 сложных (с валидным GT):")
for r in hardest[:3]:
    print(f"  idx={r['idx']}  IoU={r['iou']:.4f}  FP={r['fp']}  FN={r['fn']}  gt_area={r['gt_area']}")

error_analysis = {
    "n_images":      len(errors),
    "n_anomalies":   len(anomalies),
    "n_valid":       len(valid_rows),
    "anomaly_idx":   [r["idx"] for r in anomalies],
    "hardest":       hardest,
}
with open(ARTIFACTS / "error_analysis.json", "w") as f:
    json.dump(error_analysis, f, indent=2)
print("saved:", ARTIFACTS / "error_analysis.json")

всего val: 736
аномалий разметки (GT pet < 100 px): 1
валидных для анализа: 735
Топ-3 сложных (с валидным GT):
  idx=458  IoU=0.2021  FP=4853  FN=727  gt_area=2140
  idx=59  IoU=0.2132  FP=7410  FN=2362  gt_area=5010
  idx=127  IoU=0.4558  FP=2506  FN=993  gt_area=3924
saved: c:\Users\Lenovo\Projects\PythonSOTA\lab2\artifacts\error_analysis.json


### Поклассовая динамика аугментации

*Вес: 1 балл.*

**Постановка.** Средний IoU не показывает, каким классам помогло отражение.

Выполните следующие действия:
1. Сопоставьте поклассовый validation IoU исходной и дополненной моделей.
2. Сохраните изменения по каждому классу, не используя test для выбора модели.

**Результат.** class_deltas.json с наблюдаемыми изменениями IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [26]:
class_names = ["pet", "background", "outline"]
_, pc_base, _ = evaluate(model, val_loader, DEVICE)
_, pc_aug,  _ = evaluate(model_aug, val_loader_aug, DEVICE)

deltas = {}
for i, name in enumerate(class_names):
    deltas[name] = {
        "baseline": round(float(pc_base[i]), 4),
        "aug": round(float(pc_aug[i]), 4),
        "delta": round(float(pc_aug[i] - pc_base[i]), 4),
    }

print(json.dumps(deltas, indent=2))
with open(ARTIFACTS / "class_deltas.json", "w") as f:
    json.dump(deltas, f, indent=2)

{
  "pet": {
    "baseline": 0.8712,
    "aug": 0.8709,
    "delta": -0.0003
  },
  "background": {
    "baseline": 0.9332,
    "aug": 0.9334,
    "delta": 0.0002
  },
  "outline": {
    "baseline": 0.5154,
    "aug": 0.5197,
    "delta": 0.0043
  }
}


### Творческий бонус: диагностические маски

*Вес: 1 дополнительный балл.*

**Постановка.** Ошибки на границах полезно изучать в пространстве изображения.

Выполните следующие действия:
1. Найдите сложнейшее validation-изображение по IoU.
2. Сохраните реальную RGB-картинку, истинную и предсказанную маски для визуального разбора.

**Результат.** bonus_hard_case.png и bonus_hard_case.json с индексом и IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [27]:
import matplotlib.pyplot as plt

# Берём worst среди валидных (gt_area >= 100), а не среди аномалий
worst = hardest[0]
idx = worst["idx"]

x, y = val_ds[idx]
x_in = x.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    pred = model(x_in)["out"].argmax(dim=1).squeeze(0).cpu()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
rgb = (x.cpu() * std + mean).clamp(0,1).permute(1,2,0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12,4))
axes[0].imshow(rgb); axes[0].set_title("RGB"); axes[0].axis("off")
axes[1].imshow(y.numpy(), cmap="tab10", vmin=0, vmax=2); axes[1].set_title("GT mask"); axes[1].axis("off")
axes[2].imshow(pred.numpy(), cmap="tab10", vmin=0, vmax=2); axes[2].set_title("Pred mask"); axes[2].axis("off")
plt.tight_layout()
plt.savefig(ARTIFACTS / "bonus_hard_case.png", dpi=150)
plt.close()

bonus_info = {
    "val_idx": int(idx),
    "iou":     round(worst["iou"], 4),
    "fp":      int(worst["fp"]),
    "fn":      int(worst["fn"]),
    "gt_area": int(worst["gt_area"]),
}
with open(ARTIFACTS / "bonus_hard_case.json", "w") as f:
    json.dump(bonus_info, f, indent=2)
print("Сохранено:", bonus_info)

Сохранено: {'val_idx': 458, 'iou': 0.2021, 'fp': 4853, 'fn': 727, 'gt_area': 2140}


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Анализ результатов

**Бюджет:** LRASPP-MobileNetV3, 8 эпох, AdamW 1e-4, 256×256, SEED=42, одинаковый init для обеих моделей.

| Метрика | Baseline | + Flip | Δ |
|---|---|---|---|
| Val mIoU | 0.7732 | 0.7747 | +0.0015 |
| Test mIoU | 0.7768 | 0.7808 | +0.0040 |
| Test outline IoU | 0.5313 | 0.5410 | +0.0097 |
| Small-obj fg IoU (val) | 0.7656 | 0.7678 | +0.0022 |

- `background > pet > outline` — как предсказывает теория: контур теряется на downsampling.
- Test ≈ val (gap +0.004…+0.006) — обобщение стабильно, test не влиял на выбор модели.
- Flip помогает точечно: `pet`/`bg` ≈ 0, весь выигрыш на `outline` (+0.0097 test).
- Малые объекты (fg IoU 0.77 против 0.87) — аугментация масштабную проблему не решает.
- **Контроль init решающий:** без `torch.manual_seed(SEED)` дельта flip была −0.0034 — шум (~±0.005) перекрывал эффект.
- 1 аномалия разметки (`gt_area < 100`) отфильтрована; худшие валидные случаи — режим FP ≫ FN (модель «заливает» фон).

## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.


## Выводы

1. Baseline даёт val/test mIoU 0.77/0.78 — в ожидаемом диапазоне. Слабость — `outline` (IoU 0.52): типичный предел pixel-wise CE без boundary-aware компонент.
2. Малые объекты хуже среднего (0.77 vs 0.87) — потеря деталей на downsampling backbone.
3. Синхронный flip реализован корректно и даёт устойчивый +0.004 test mIoU, сконцентрированный на `outline` (+0.0097) — там, где предсказывает теория.
4. Ablation без контроля init даёт ложный отрицательный результат: seed-control обязателен при сравнении конфигураций.
5. Все артефакты сохранены в `artifacts/`.